# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema at:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed (run if not yet installed)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Show dataset summary
print(f"{metadata.name}: {metadata.description}")
print(f"License: {metadata.license}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
In this section, we examine the available record sets and their fields using their unique `@id`s.

> In Croissant, the `recordSet` is the entity grouping related records (typically corresponding to one table of data); every field and column used should be referenced by its `@id`. Listing the available record sets and their schema helps guide the rest of our exploration.

In [ ]:
# Get all record sets by @id
print("Available record sets (@id):")
for record_set in metadata.recordSet:
    print(f"- {record_set['@id']}")

# For demonstration, list their associated fields and field @id's
for record_set in metadata.recordSet:
    print(f"\nRecord set: {record_set['@id']}")
    if 'field' in record_set:
        if isinstance(record_set['field'], list):
            for field in record_set['field']:
                print(f"  - Field: {field['@id']} (name: {field.get('name', 'N/A')})")
        else:
            field = record_set['field']
            print(f"  - Field: {field['@id']} (name: {field.get('name', 'N/A')})")
    else:
        print("  (No fields found)")

## 3. Data Extraction
Let's select a record set to extract records as a DataFrame. All references are made using the `@id` as shown above.

For this dataset, let's extract records from the principal tabular record set. You should adjust the `record_sets_ids` variable to match actual record set `@id`s.

In [ ]:
# Specify target record set(s) from the metadata above
# For this dataset, it's typically something like:
#   'https://api.app.sen.science/frontiers/7862866/555dc17f-14a2-49e5-9377-ed52246a1657'
# Replace this @id if different based on the output from the Data Overview step

record_sets_ids = [rs['@id'] for rs in metadata.recordSet]
dataframes = dict()

for record_set_id in record_sets_ids:
    print(f"\nExtracting records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Fields (@id) for {record_set_id}:")
    print(list(df.columns))
    print(df.head(3))

## 4. Exploratory Data Analysis (EDA)

Now let's perform some simple EDA steps. We'll:
- Filter by a numeric field (referenced by `@id`),
- Normalize the numeric field,
- Group by a categorical field (referenced by `@id`).

Identify a numeric field (e.g., age) and a group (categorical) field from the previous extraction.

In [ ]:
# Choose the record set that contains clinical variables.
# Replace these @id's with your actual field @id's for age, sex, etc. Found in the previous step.

# Example: Let's assume the following for demonstration (update as per actual field @id's!):
target_record_set_id = record_sets_ids[0]  # or specify directly
df = dataframes[target_record_set_id]

# Find a numeric field (e.g. 'age' by its @id), and a group field @id (e.g. 'sex')

print('Available columns / field @id in selected record set:')
for col in df.columns:
    print(f"- {col}")

# (Edit to the appropriate field @id for numeric_field, group_field below)
numeric_field_id = [col for col in df.columns if 'age' in col.lower()][0]  # e.g., '@id:age' or similar
group_field_id = [col for col in df.columns if 'sex' in col.lower() or 'gender' in col.lower()]
group_field_id = group_field_id[0] if group_field_id else None

print(f"\nUsing numeric field: {numeric_field_id}; group field: {group_field_id}")

# Convert the numeric column to numeric (if needed)
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter records based on a threshold (e.g., age > 40)
threshold = 40
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"\nFiltered records where {numeric_field_id} > {threshold} (N={len(filtered_df)}):")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field (e.g., sex or other group_field)
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
    print(grouped_df)

## 5. Visualization

Let's visualize the distribution of the numeric field and the group-wise differences.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to:
- Load a Croissant-encoded FAIR² dataset using `mlcroissant`;
- Identify and extract data from available record sets and fields using their `@id` references;
- Perform basic cleaning, filtering, normalization and group-wise EDA;
- Visualize key numeric fields and grouped distributions.

These steps can be extended for more complex processing and modeling. Remember to always reference entities by their precise Croissant `@id` for full reproducibility and transparency.